# pypgo.fem & pypgo.energy — Deformation FEM API Demo

This tutorial demonstrates how to build deformation energies for FEM
simulations using `pypgo.fem` and evaluate them through
`pypgo.energy`, with **real mesh assets** (tet, cubic, shell).

**Audience:** users building FEM simulations with tet, hex, or shell
elements in pypgo.

**Prerequisites:** familiarity with `pypgo.energy` basics (see
`energy_api_demo.ipynb`) and with NumPy.

**Learning goals:**

1. Load `.veg` / `.obj` assets and create `SimulationMesh` objects.
2. Choose a formulation (`TetP1`, `LinearCubic`, `TricubicHermite`, `KoiterShell`).
3. Choose elastic and plastic material laws.
4. Build a `DeformationModelState`, then a `DeformationEnergy`.
5. Evaluate energy, gradient, Hessian at displacement states.
6. Compose with `EnergySet` for multi-term objective functions.
7. Understand `state_kind` and `rest_position` conventions.


## Outline

1. Setup — imports and asset discovery
2. Tet P1 energy from `bunny.veg` (711 vertices, 436 elements)
3. Cubic hex trilinear energy from `box.veg` (125 vertices, 64 elements)
4. Shell Koiter energy from `shell.obj` (surface mesh)
4b. Tricubic Hermite from `box.veg` (24 DOFs/vertex, C1, patch tests)
5. Evaluate energy, gradient, Hessian at zero state
6. Compose with EnergySet + VertexAttachment
7. Lifetime and ownership
8. Exercise


In [53]:
from pathlib import Path

import numpy as np
import pypgo as pgo
import pypgo.fem as pf
import pypgo.energy as pe
from pypgo.mesh.volume import VolumeMesh, read_veg


## 1. Setup — imports and asset discovery

Assets live under `pypgo/examples/assets/`:
- `veg/tet/*.veg` — tetrahedral meshes with ENu material payloads
- `veg/cubic/*.veg` — hexahedral (cubic) meshes with ENu material payloads
- `obj/*.obj` — triangle surface meshes (for shell)

All `.veg` files carry ENu material, so `StableNeo` / `StVK` /
`LinearElastic` laws are all valid.


In [54]:
def _repo_root() -> Path:
    for p in Path.cwd().resolve().parents:
        if (p / ".git").exists():
            return p
    return Path.cwd().resolve()

ROOT = _repo_root()
ASSETS = ROOT / "examples" / "assets"
TET_VEG = ASSETS / "veg" / "tet"
CUBIC_VEG = ASSETS / "veg" / "cubic"
OBJ_DIR = ASSETS / "obj"

print("Repo root:", ROOT)
print()
print("Tet .veg assets:  ", sorted(f.name for f in TET_VEG.glob("*.veg")))
print("Cubic .veg assets:", sorted(f.name for f in CUBIC_VEG.glob("*.veg")))
print("OBJ assets:       ", sorted(f.name for f in OBJ_DIR.glob("*.obj")))


Repo root: /Users/jinceyang/Desktop/codebase/libpgo

Tet .veg assets:   ['box-with-sphere.veg', 'box.veg', 'bunny.veg', 'dragon.veg', 'torus.veg']
Cubic .veg assets: ['box-with-sphere.veg', 'box.veg', 'bunny.veg', 'dragon.veg']
OBJ assets:        ['bottom.obj', 'box-with-sphere.obj', 'box.obj', 'bunny.obj', 'dragon.obj', 'shell.obj']


## 2. Tet P1 energy from `bunny.veg`

We load a real tetrahedral mesh — `bunny.veg` (711 vertices, 436
tets, ENu material) — convert it to a `VolumeMesh`, then to a
solver-ready `SimulationMesh`, and finally build a
`DeformationEnergy` with the `TetP1` formulation.

**Steps:** `.veg` → `VegFile` → `VolumeMesh` → `SimulationMesh` →
`deformation_model_state()` and `deformation_energy()`


In [55]:
# Load bunny.veg tet mesh
bunny_path = str(TET_VEG / "bunny.veg")
veg = read_veg(bunny_path)

# Convert to VolumeMesh (extracts tet geometry + ENu material)
bunny_volume = VolumeMesh.from_veg_file(veg)
print(f"VolumeMesh: {bunny_volume.num_vertices} vertices, "
      f"{bunny_volume.num_elements} elements")
print(f"mesh_type: {bunny_volume.mesh_type}")


VolumeMesh: 203 vertices, 579 elements
mesh_type: MeshType.Tet


In [56]:
# Create solver-ready simulation mesh
bunny_mesh = pgo.fem.SimulationMesh.create_volumetric(bunny_volume)
print(f"mesh_type:      {bunny_mesh.mesh_type}")
print(f"num_vertices:   {bunny_mesh.num_vertices}")
print(f"num_elements:   {bunny_mesh.num_elements}")
print(f"expected DOFs:  {bunny_mesh.num_vertices * 3}")


mesh_type:      tet
num_vertices:   203
num_elements:   579
expected DOFs:  609


In [57]:
# Build Tet P1 deformation energy
bunny_state = pf.deformation_model_state(
    bunny_mesh,
    elastic=pf.StableNeo(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=6),
    plastic_field=pf.ElementwiseField(),
)
energy_tet = pf.deformation_energy(
    bunny_state,
    formulation=pf.TetP1(),
)
nv, ne = bunny_mesh.num_vertices, bunny_state.num_elements
print(type(energy_tet).__name__)
print(f"  deformation DOFs:     {nv} verts × 3 = {nv * 3}")
print(f"  elastic params:       {energy_tet.num_elastic_params} / elem ({energy_tet.num_elastic_dofs} total)")
print(f"  plastic params:       {energy_tet.num_plastic_params} / elem ({energy_tet.num_plastic_dofs} total)")
print(f"  energy.num_dofs:      {energy_tet.num_dofs}  (deformation only)")
print(f"  state_kind:           {energy_tet.state_kind}")
print(repr(energy_tet))


[2026-06-08 11:49:18.107] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: tet_p1 and deformation model state
[2026-06-08 11:49:18.107] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.109] [info] [deformationModelAssembler.cpp:90] Assembler parameter:0,6
DeformationEnergy
  deformation DOFs:     203 verts × 3 = 609
  elastic params:       0 / elem (0 total)
  plastic params:       6 / elem (3474 total)
  energy.num_dofs:      609  (deformation only)
  state_kind:           displacement
DeformationEnergy(609 DOFs, state_kind='displacement')


The formulation is passed explicitly.  Cubic and shell examples use
their topology-specific formulations below.


In [58]:
# Also try with StVK and different plastic DOFs
bunny_state_stvk = pf.deformation_model_state(
    bunny_mesh,
    elastic=pf.StVK(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=3),
    plastic_field=pf.ElementwiseField(),
)
energy_tet_stvk = pf.deformation_energy(
    bunny_state_stvk,
    formulation=pf.TetP1(),
)
nv, ne = bunny_mesh.num_vertices, bunny_state_stvk.num_elements
print(f"StVK + dof3:")
print(f"  deformation DOFs:     {nv} verts × 3 = {nv * 3}")
print(f"  elastic params:       {energy_tet_stvk.num_elastic_params} / elem ({energy_tet_stvk.num_elastic_dofs} total)")
print(f"  plastic params:       {energy_tet_stvk.num_plastic_params} / elem ({energy_tet_stvk.num_plastic_dofs} total)")
print(f"  energy.num_dofs:      {energy_tet_stvk.num_dofs}  (deformation only)")


[2026-06-08 11:49:18.122] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: tet_p1 and deformation model state
[2026-06-08 11:49:18.122] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.125] [info] [deformationModelAssembler.cpp:90] Assembler parameter:0,3
StVK + dof3:
  deformation DOFs:     203 verts × 3 = 609
  elastic params:       0 / elem (0 total)
  plastic params:       3 / elem (1737 total)
  energy.num_dofs:      609  (deformation only)


## 3. Cubic hex trilinear energy from `box.veg`

Cubic (hex) meshes use 8-node trilinear elements.  The formulation
`LinearCubic()` is **required** — the factory will raise
`ValueError` if you omit it.

We use `box.veg` (125 vertices, 64 hex elements) from the cubic
asset directory.


In [59]:
# Load box.veg cubic mesh
box_path = str(CUBIC_VEG / "box.veg")
box_veg = read_veg(box_path)
box_volume = VolumeMesh.from_veg_file(box_veg)
print(f"VolumeMesh: {box_volume.num_vertices} vertices, "
      f"{box_volume.num_elements} elements")
print(f"mesh_type: {box_volume.mesh_type}")


VolumeMesh: 125 vertices, 64 elements
mesh_type: MeshType.Cubic


In [60]:
box_mesh = pgo.fem.SimulationMesh.create_volumetric(box_volume)
print(f"mesh_type:    {box_mesh.mesh_type}")
print(f"num_vertices: {box_mesh.num_vertices}")
print(f"num_elements: {box_mesh.num_elements}")


mesh_type:    cubic
num_vertices: 125
num_elements: 64


In [61]:
# REQUIRED: explicit LinearCubic() formulation
box_state = pf.deformation_model_state(
    box_mesh,
    elastic=pf.StableNeo(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=6),
    plastic_field=pf.ElementwiseField(),
)
energy_cubic = pf.deformation_energy(
    box_state,
    formulation=pf.LinearCubic(),
)
nv = box_mesh.num_vertices
print(f"  deformation DOFs:     {nv} verts × 3 = {nv * 3}")
print(f"  elastic params:       {energy_cubic.num_elastic_params} / elem ({energy_cubic.num_elastic_dofs} total)")
print(f"  plastic params:       {energy_cubic.num_plastic_params} / elem ({energy_cubic.num_plastic_dofs} total)")
print(f"  energy.num_dofs:      {energy_cubic.num_dofs}")
print(f"  state_kind:           {energy_cubic.state_kind}")
print(f"  rest_position shape:  {energy_cubic.rest_position.shape}")


[2026-06-08 11:49:18.148] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: hex_trilinear and deformation model state
[2026-06-08 11:49:18.148] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.149] [info] [deformationModelAssembler.cpp:90] Assembler parameter:0,6
  deformation DOFs:     125 verts × 3 = 375
  elastic params:       0 / elem (0 total)
  plastic params:       6 / elem (384 total)
  energy.num_dofs:      375
  state_kind:           displacement
  rest_position shape:  (125, 3)


In [62]:
# Omitting the formulation → ValueError
try:
    pf.deformation_energy(
        box_state,
    )
except ValueError as e:
    print(f"Error (expected): {e}")


Error (expected): formulation is required. Pass TetP1(), LinearCubic(), TricubicHermite(), or KoiterShell().


## 4. Shell Koiter energy from `shell.obj`

Shell meshes use the Koiter thin-shell formulation.  We load
`shell.obj` (a triangulated surface), create a `TriMeshData`,
assign a `KoiterStVKShellMaterial`, and build the energy.

Shells require **shell-specific** elastic/plastic wrappers:
`KoiterStVK()` and `ShellPlasticity(dofs=1)`.


In [63]:
# Load shell.obj → TriMeshData
shell_path = str(OBJ_DIR / "shell.obj")
shell_surface = pgo.mesh.read_obj(shell_path)
print(f"Surface: {shell_surface.num_vertices} vertices, "
      f"{shell_surface.num_elements} triangles")

# Assign shell material (membrane + bending stiffness)
shell_mat = pgo.fem.KoiterStVKShellMaterial(
    thickness=0.001,
    E_membrane=2e6,
    nu_membrane=0.35,
)
shell_mesh = pgo.fem.SimulationMesh.create_shell(shell_surface, shell_mat)
print(f"mesh_type:    {shell_mesh.mesh_type}")
print(f"num_vertices: {shell_mesh.num_vertices}")


Surface: 1089 vertices, 2048 triangles
mesh_type:    shell
num_vertices: 1089


In [64]:
# Build shell deformation energy
shell_state = pf.deformation_model_state(
    shell_mesh,
    elastic=pf.KoiterStVK(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.ShellPlasticity(dofs=1),
    plastic_field=pf.ElementwiseField(),
)
energy_shell = pf.deformation_energy(
    shell_state,
    formulation=pf.KoiterShell(),
)
nv = shell_mesh.num_vertices
print(f"  deformation DOFs:     {nv} verts × 3 = {nv * 3}")
print(f"  elastic params:       {energy_shell.num_elastic_params} / elem ({energy_shell.num_elastic_dofs} total)")
print(f"  plastic params:       {energy_shell.num_plastic_params} / elem ({energy_shell.num_plastic_dofs} total)")
print(f"  energy.num_dofs:      {energy_shell.num_dofs}")
print(f"  state_kind:           {energy_shell.state_kind}")
print(repr(energy_shell))


[2026-06-08 11:49:18.173] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: shell_koiter and deformation model state
[2026-06-08 11:49:18.173] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.174] [info] [deformationModelAssembler.cpp:90] Assembler parameter:5,1
  deformation DOFs:     1089 verts × 3 = 3267
  elastic params:       5 / elem (10240 total)
  plastic params:       1 / elem (2048 total)
  energy.num_dofs:      3267
  state_kind:           displacement
DeformationEnergy(3267 DOFs, state_kind='displacement')


## 4b. Tricubic Hermite — 24 DOFs per vertex (C1)

`TricubicHermite()` is a high-order hex formulation where each vertex
carries **8 Hermite modes** (value + 7 derivative modes), yielding
**24 DOFs per vertex** instead of the usual 3.  The field is C1
continuous within each cell.

It currently targets **regular axis-aligned hex grids** — the
formulation's DOF layout assumes the local ξ/η/ζ axes align with
global x/y/z, so `box.veg` (5×5×5 uniform grid) is the ideal test
case.  General curvilinear meshes need the next-phase inverse-design
transform.


In [65]:
# Load the regular-grid cubic box (required for Hermite MVP)
herm_path = str(CUBIC_VEG / "box.veg")
herm_veg = read_veg(herm_path)
herm_volume = VolumeMesh.from_veg_file(herm_veg)
herm_mesh = pgo.fem.SimulationMesh.create_volumetric(herm_volume)
print(f"VolumeMesh: {herm_volume.num_vertices} vertices, "
      f"{herm_volume.num_elements} hex elements")

# Build Hermite energy — 24 DOFs per vertex
herm_state = pf.deformation_model_state(
    herm_mesh,
    elastic=pf.StableNeo(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=6),
    plastic_field=pf.ElementwiseField(),
)
energy_herm = pf.deformation_energy(
    herm_state,
    formulation=pf.TricubicHermite(),
)
print(f"  deformation DOFs:     {herm_volume.num_vertices} verts × 24 = {energy_herm.num_dofs}")
print(f"  elastic params:       {energy_herm.num_elastic_params} / elem ({energy_herm.num_elastic_dofs} total)")
print(f"  plastic params:       {energy_herm.num_plastic_params} / elem ({energy_herm.num_plastic_dofs} total)")
print(f"  state_kind:           {energy_herm.state_kind}")
print(f"  rest_position shape:  {energy_herm.rest_position.shape}")


VolumeMesh: 125 vertices, 64 hex elements
[2026-06-08 11:49:18.210] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: hex_tricubic_hermite and deformation model state
[2026-06-08 11:49:18.210] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.259] [info] [deformationModelAssembler.cpp:90] Assembler parameter:0,6
  deformation DOFs:     125 verts × 24 = 3000
  elastic params:       0 / elem (0 total)
  plastic params:       6 / elem (384 total)
  state_kind:           displacement
  rest_position shape:  (1000, 3)


In [66]:
# Compare: same mesh with LinearCubic() → only 3 DOFs per vertex
box_state = pf.deformation_model_state(
    herm_mesh,
    elastic=pf.StableNeo(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=6),
    plastic_field=pf.ElementwiseField(),
)
energy_linear = pf.deformation_energy(
    box_state,
    formulation=pf.LinearCubic(),
)
print(f"LinearCubic:   {energy_linear.num_dofs} deformation DOFs  (= {herm_volume.num_vertices} × 3)")
print(f"TricubicHermite: {energy_herm.num_dofs} deformation DOFs  (= {herm_volume.num_vertices} × 24)")


[2026-06-08 11:49:18.421] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: hex_trilinear and deformation model state
[2026-06-08 11:49:18.421] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.422] [info] [deformationModelAssembler.cpp:90] Assembler parameter:0,6
LinearCubic:   375 deformation DOFs  (= 125 × 3)
TricubicHermite: 3000 deformation DOFs  (= 125 × 24)


In [67]:
# --- Patch tests ---

rest = np.asarray(energy_herm.rest_position)   # (nvtx, 24) → value modes are cols 0:3
rest_modes = rest.reshape(-1, 8, 3)            # per-vertex 8 modes × 3 coords

def hermite_affine_disp(A, t):
    """Map an affine deformation x = A·X + t onto Hermite DOFs.
    Value modes transform as A·pos + t; derivative modes as A·mode."""
    deformed = rest_modes @ A.T
    deformed[:, 0, :] += t                     # translation only on value mode
    return (deformed - rest_modes).reshape(-1)

u0 = energy_herm.zero_state()
print(f"rest energy:  {energy_herm.value(u0):.4e}")
print(f"rest |grad|:  {float(np.linalg.norm(energy_herm.gradient(u0))):.2e}")

# Rigid translation → zero energy
u_t = hermite_affine_disp(np.eye(3), np.array([0.3, -0.7, 1.1]))
print(f"translation:  {energy_herm.value(u_t):.4e}")

# Rigid rotation → zero energy (frame-invariant material)
ax = np.array([0.3, 0.8, 0.5]); ax /= np.linalg.norm(ax)
th = 0.4
K = np.array([[0, -ax[2], ax[1]], [ax[2], 0, -ax[0]], [-ax[1], ax[0], 0]])
R = np.eye(3) + np.sin(th) * K + (1 - np.cos(th)) * (K @ K)
u_r = hermite_affine_disp(R, np.zeros(3))
print(f"rotation:     {energy_herm.value(u_r):.4e}")

# Affine deformation → Hermite matches LinearCubic exactly
A = np.array([[1.05, 0.03, 0.0], [0.0, 0.98, 0.02], [0.01, 0.0, 1.03]])
t = np.array([0.01, -0.02, 0.0])
eH = energy_herm.value(hermite_affine_disp(A, t))
eL = energy_linear.value(
    (herm_volume.mesh_data.vertices @ A.T + t - herm_volume.mesh_data.vertices).reshape(-1))
print(f"affine Hermite:     {eH:.6e}")
print(f"affine LinearCubic: {eL:.6e}")
print(f"relative diff:      {abs(eH - eL) / max(1.0, abs(eL)):.2e}")


rest energy:  0.0000e+00
rest |grad|:  2.78e-11
translation:  5.8962e-12
rotation:     3.4128e-12
affine Hermite:     8.900723e+03
affine LinearCubic: 8.900723e+03
relative diff:      1.63e-15


### When to use TricubicHermite

- **Smooth deformation fields** — C1 continuity avoids the kinks of trilinear hex.
- **Inverse design / PDE-constrained optimization** — the richer DOF space
  parameterises a higher-dimensional design space.
- **Currently targets regular axis-aligned hex grids**; general curvilinear
  meshes are planned for a future phase.


## 5. Evaluate energy, gradient, Hessian at zero state

All energy types share the same evaluation protocol.
`state_kind == "displacement"` means the state vector `u` is a
displacement from `rest_position`.


In [68]:
# ---- Tet (bunny) ----
u_tet = energy_tet.zero_state()
print("=== Tet (bunny) ===")
print(f"zero_state:     shape={u_tet.shape}, all_zero={np.all(u_tet == 0.0)}")
print(f"energy(u=0):    {energy_tet.value(u_tet):.6e}")
g_tet = energy_tet.gradient(u_tet)
print(f"gradient norm:  {np.linalg.norm(g_tet):.6e}")
H_tet = energy_tet.hessian(u_tet)
print(f"Hessian:        shape={H_tet.shape}, nnz={H_tet.nnz}")


=== Tet (bunny) ===
zero_state:     shape=(609,), all_zero=True
energy(u=0):    5.436119e-16
gradient norm:  2.236764e-13
Hessian:        shape=(609, 609), nnz=18837


In [69]:
# Energy increases under a perturbation (tet)
u_pert = u_tet.copy()
u_pert[3] = 0.01   # perturb a vertex x-displacement
print(f"energy(u=0):              {energy_tet.value(u_tet):.6e}")
print(f"energy(perturbed):        {energy_tet.value(u_pert):.6e}")
print(f"increase:                 {energy_tet.value(u_pert) - energy_tet.value(u_tet):.6e}")


energy(u=0):              5.436119e-16
energy(perturbed):        2.999927e-01
increase:                 2.999927e-01


In [70]:
# ---- Cubic (box) ----
u_cubic = energy_cubic.zero_state()
print("=== Cubic (box) ===")
print(f"energy(u=0):    {energy_cubic.value(u_cubic):.6e}")
g_cubic = energy_cubic.gradient(u_cubic)
print(f"gradient norm:  {np.linalg.norm(g_cubic):.6e}")
H_cubic = energy_cubic.hessian(u_cubic)
print(f"Hessian:        shape={H_cubic.shape}, nnz={H_cubic.nnz}")


=== Cubic (box) ===
energy(u=0):    0.000000e+00
gradient norm:  2.829719e-27
Hessian:        shape=(375, 375), nnz=19773


In [71]:
# ---- Shell ----
u_shell = energy_shell.zero_state()
print("=== Shell ===")
print(f"energy(u=0):    {energy_shell.value(u_shell):.6e}")
g_shell = energy_shell.gradient(u_shell)
print(f"gradient norm:  {np.linalg.norm(g_shell):.6e}")
H_shell = energy_shell.hessian(u_shell)
print(f"Hessian:        shape={H_shell.shape}, nnz={H_shell.nnz}")


=== Shell ===
energy(u=0):    0.000000e+00
gradient norm:  0.000000e+00
Hessian:        shape=(3267, 3267), nnz=174483


### Rest position and state convention

Each `DeformationEnergy` exposes the undeformed vertex positions
as a `(num_vertices, 3)` ndarray.


In [72]:
for label, e in [("Tet (bunny)", energy_tet),
                  ("Cubic (box)", energy_cubic),
                  ("Shell",      energy_shell)]:
    rp = e.rest_position
    print(f"{label:15s} rest_pos={rp.shape}, "
          f"vertices={e.num_vertices}, "
          f"state_kind={e.state_kind}, "
          f"isinstance(PotentialEnergy)={isinstance(e, pe.PotentialEnergy)}")


Tet (bunny)     rest_pos=(203, 3), vertices=203, state_kind=displacement, isinstance(PotentialEnergy)=True
Cubic (box)     rest_pos=(125, 3), vertices=125, state_kind=displacement, isinstance(PotentialEnergy)=True
Shell           rest_pos=(1089, 3), vertices=1089, state_kind=displacement, isinstance(PotentialEnergy)=True


## 6. Compose with EnergySet + VertexAttachment

`DeformationEnergy` is fully compatible with `EnergySet`.  Here we
combine the tet deformation energy with an external gravity-like
force and a soft pin constraint on selected vertices.


In [73]:
# ---- External force (gravity-like, z-direction) ----
n = energy_tet.num_dofs
gravity = np.zeros(n, dtype=np.float64)
gravity[2::3] = -9.81
force = pe.LinearEnergy(gravity)

# ---- Pin a few vertices (simulate fixed boundary) ----
# Pin vertices 0, 1, 2 to their rest positions
pin_vtx = np.array([0, 1, 2], dtype=np.int64)
rp = energy_tet.rest_position
pin_targets = rp[pin_vtx].ravel()  # flatten to (m*3,)

pin = pe.VertexAttachment(
    sim_mesh=bunny_mesh,
    vertex_indices=pin_vtx,
    target_positions=pin_targets,
    coeff=1e6,
)
print(f"Pin constraint: {pin.num_dofs} DOFs, state_kind={pin.state_kind}")


Pin constraint: 609 DOFs, state_kind=displacement


In [74]:
# ---- Combine in EnergySet ----
total = pe.EnergySet([
    (energy_tet,  1.0),    # deformation
    (force,      -1.0),    # external work
    (pin,         1.0),    # soft pin constraint
])
print(repr(total))


EnergySet(3 terms, 609 DOFs, state_kind='generic')


In [75]:
# Evaluate total energy at zero displacement
u = total.zero_state()
print(f"total energy(u=0): {total.value(u):.6e}")

# Perturb — energy increases due to all three terms
u_pert = u.copy()
u_pert[3:6] = 0.005
print(f"total energy(perturbed): {total.value(u_pert):.6e}")

# Adjust weights at runtime
total.set_weight(1, 0.0)   # disable gravity
print(f"gravity disabled:        {total.value(u_pert):.6e}")
total.set_weight(1, -1.0)  # re-enable
print(f"gravity re-enabled:      {total.value(u_pert):.6e}")


total energy(u=0): 3.843675e+04
total energy(perturbed): 3.825825e+04
gravity disabled:        3.825820e+04
gravity re-enabled:      3.825825e+04


## 7. Lifetime and ownership

The C++ layer owns the mesh and energy data via `shared_ptr`.
You can delete the Python `SimulationMesh` wrapper — the energy
still works.


In [76]:
import gc

# Build energy from a temporary mesh
tmp_veg = read_veg(str(TET_VEG / "box.veg"))
tmp_vol = VolumeMesh.from_veg_file(tmp_veg)
tmp_sim = pgo.fem.SimulationMesh.create_volumetric(tmp_vol)

tmp_state = pf.deformation_model_state(
    tmp_sim,
    elastic=pf.StableNeo(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=6),
    plastic_field=pf.ElementwiseField(),
)
e = pf.deformation_energy(
    tmp_state,
    formulation=pf.TetP1(),
)
u0 = e.zero_state()
val_before = e.value(u0)

# Delete Python mesh wrappers and state wrapper
del tmp_state, tmp_sim, tmp_vol, tmp_veg
gc.collect()

val_after = e.value(u0)
print(f"value before mesh deletion: {val_before:.10e}")
print(f"value after mesh deletion:  {val_after:.10e}")
print(f"difference:                 {abs(val_before - val_after):.2e}")
assert abs(val_before - val_after) < 1e-14
print("✓ energy survives mesh deletion")


[2026-06-08 11:49:18.666] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: tet_p1 and deformation model state
[2026-06-08 11:49:18.666] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.667] [info] [deformationModelAssembler.cpp:90] Assembler parameter:0,6
value before mesh deletion: 1.0473328116e-11
value after mesh deletion:  1.0473328116e-11
difference:                 1.62e-27
✓ energy survives mesh deletion


### Two independent energies from the same mesh

The same `SimulationMesh` can be used to build multiple energies
— it's borrowed (not consumed) by the factory.


In [77]:
state1 = pf.deformation_model_state(
    bunny_mesh,
    elastic=pf.StableNeo(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=6),
    plastic_field=pf.ElementwiseField(),
)
state2 = pf.deformation_model_state(
    bunny_mesh,
    elastic=pf.StVK(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=3),
    plastic_field=pf.ElementwiseField(),
)
e1 = pf.deformation_energy(state1, formulation=pf.TetP1())
e2 = pf.deformation_energy(state2, formulation=pf.TetP1())
u = e1.zero_state()
print(f"e1 (StableNeo, dof6):  {e1.value(u):.6e}")
print(f"e2 (StVK, dof3):       {e2.value(u):.6e}")
print(f"both use same mesh ✓")


[2026-06-08 11:49:18.760] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: tet_p1 and deformation model state
[2026-06-08 11:49:18.760] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.762] [info] [deformationModelAssembler.cpp:90] Assembler parameter:0,6
[2026-06-08 11:49:18.774] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: tet_p1 and deformation model state
[2026-06-08 11:49:18.774] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.776] [info] [deformationModelAssembler.cpp:90] Assembler parameter:0,3
e1 (StableNeo, dof6):  5.436119e-16
e2 (StVK, dof3):       1.510834e-15
both use same mesh ✓


## 8. Exercise

Use the tet `box.veg` asset:
1. Build a `DeformationEnergy` with `StVK` material and `dofs=3`
   plastic.
2. Verify the Hessian at rest is symmetric (check $\|H - H^T\| < 10^{-8}$).
3. Add a `VertexAttachment` that pins the first 5 vertices.
4. Compose them in an `EnergySet`.
5. Disable the pin constraint weight → energy drops.


In [78]:
# Your solution here
box_tet_path = str(TET_VEG / "box.veg")
box_tet_veg = read_veg(box_tet_path)
box_tet_vol = VolumeMesh.from_veg_file(box_tet_veg)
box_tet_sim = pgo.fem.SimulationMesh.create_volumetric(box_tet_vol)

box_tet_state = pf.deformation_model_state(
    box_tet_sim,
    elastic=pf.StVK(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=3),
    plastic_field=pf.ElementwiseField(),
)
e = pf.deformation_energy(
    box_tet_state,
    formulation=pf.TetP1(),
)
u = e.zero_state()
print(f"energy at rest: {e.value(u):.6e}")

# Check Hessian symmetry
H = e.hessian(u)
rows, cols, vals = H.to_coo()
H_dense = np.zeros(H.shape, dtype=np.float64)
for r, c, v in zip(rows, cols, vals):
    H_dense[r, c] = v
sym_err = np.max(np.abs(H_dense - H_dense.T))
print(f"Hessian symmetry error: {sym_err:.2e}")
assert sym_err < 1e-8, "Hessian must be symmetric"
print("✓ Hessian is symmetric")

# Pin first 5 vertices
rp = e.rest_position
pin_vtx = np.arange(5, dtype=np.int64)
pin = pe.VertexAttachment(
    sim_mesh=box_tet_sim,
    vertex_indices=pin_vtx,
    target_positions=rp[pin_vtx].ravel(),
    coeff=1e6,
)

total = pe.EnergySet([(e, 1.0), (pin, 1.0)])
print(f"total energy (with pin): {total.value(u):.6e}")

total.set_weight(1, 0.0)  # disable pin
print(f"total energy (no pin):  {total.value(u):.6e}")


[2026-06-08 11:49:18.793] [info] [deformationEnergyBuilder.cpp:29] Building deformation energy with formulation: tet_p1 and deformation model state
[2026-06-08 11:49:18.793] [info] [deformationModelManager.cpp:283] Initializing element models (manager path)...
[2026-06-08 11:49:18.795] [info] [deformationModelAssembler.cpp:90] Assembler parameter:0,3
energy at rest: 4.206332e-11
Hessian symmetry error: 9.31e-10
✓ Hessian is symmetric
total energy (with pin): 5.918402e+05
total energy (no pin):  4.206332e-11


## Available formulations, materials, and plastics

| Type | Formulation | Nodes | DOFs/vertex | Example |
|---|---|---|---|---|
| Tet | `pf.TetP1()` (default) | 4 | 3 | `veg/tet/*.veg` |
| Cubic (trilinear) | `pf.LinearCubic()` (required) | 8 | 3 | `veg/cubic/*.veg` |
| Cubic (tricubic Hermite) | `pf.TricubicHermite()` (required) | 8 | 24 | `veg/cubic/*.veg` (regular grid) |
| Shell | `pf.KoiterShell()` (required) | 6 | 3 | `obj/shell.obj` |

| Elastic Law | Wrapper | Valid With |
|---|---|---|
| Stable Neo-Hookean | `pf.StableNeo()` | ENu payload |
| StVK | `pf.StVK()` | ENu payload |
| StVK Volume | `pf.StVKVolume()` | ENu payload |
| Linear | `pf.LinearElastic()` | ENu payload |
| Mooney-Rivlin | `pf.MooneyRivlin()` | MooneyRivlin payload |
| Koiter StVK | `pf.KoiterStVK()` | Shell ENuh payload |

| Plastic | Wrapper | Valid With |
|---|---|---|
| Volumetric (6 dof) | `pf.VolumetricPlasticity(dofs=6)` | Tet, Cubic |
| Volumetric (3 dof) | `pf.VolumetricPlasticity(dofs=3)` | Tet, Cubic |
| Volumetric (0 dof) | `pf.VolumetricPlasticity(dofs=0)` | Tet, Cubic |
| Shell (1 dof) | `pf.ShellPlasticity(dofs=1)` | Shell |
| Shell (0 dof) | `pf.ShellPlasticity(dofs=0)` | Shell |

**Pitfall:** Cubic and shell meshes must pass an explicit
formulation.  Missing formulation → `ValueError`.

**Pitfall:** Deformation energy state is always displacement.
Passing absolute positions will give wrong results.

**Note:** `VolumeMesh.from_veg_file()` auto-detects the element
type from the `.veg` file.  Make sure you load from the correct
`veg/tet/` or `veg/cubic/` subdirectory so the mesh type matches
your formulation.

**Extension:** Combine `DeformationEnergy` with
`VertexAttachment` and `EnergySet` for full static/dynamic
IPC simulations.

**Heavier assets:** `dragon.veg` (1247 vertices) and
`box-with-sphere.veg` are available in both `veg/tet/` and
`veg/cubic/` for experimentation.
